<a href="https://colab.research.google.com/github/sjayavelu73/langgraph/blob/lang1/React.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
from typing import Annotated, Sequence, TypedDict
from dotenv import load_dotenv
from langchain_core.messages import BaseMessage,ToolMessage,SystemMessage,AIMessage,HumanMessage
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("ai_agents_openai")

class AgentState(TypedDict):
  messages: Annotated[Sequence[BaseMessage], add_messages]

@tool
def add(a: int, b: int) -> int:
  """Addition tool that adds two numbers together."""
  return a + b

tools = [add]
model = ChatOpenAI(model="gpt-4o").bind_tools(tools)

def model_call(state:AgentState) -> AgentState:
  system_prompt = SystemMessage(content= "You are my AI assistant, please answer my query to the best of your ability." )
  response = model.invoke([system_prompt] + state["messages"])
  return {"messages": [response]}

def should_continue(state: AgentState):
  messages = state["messages"]
  last_message = messages[-1]
  if not last_message.tool_calls:
    return "end"
  else: return "continue"

graph = StateGraph(AgentState)
graph.add_node("our_agent", model_call)

toolnode=ToolNode(tools)
graph.add_node("tools",toolnode)
graph.set_entry_point("our_agent")
graph.add_conditional_edges("our_agent",
                            should_continue,
                           {"continue":"tools","end":END}

)
graph.add_edge("tools", "our_agent")

app=graph.compile()

inputs={"messages":[("user","Add 13 to 25,Integral of logx, 89+100")]}
msg=app.stream(inputs,stream_mode="values")
for s in msg:
  message=s["messages"][-1]
  if isinstance(message,tuple):
    print(message)
  else:
    message.pretty_print()



================================ Human Message =================================

Add 13 to 25,Integral of logx, 89+100
================================== Ai Message ==================================
Tool Calls:
  add (call_B6Cvgdo3slrBEX9wh3vmd9GQ)
 Call ID: call_B6Cvgdo3slrBEX9wh3vmd9GQ
  Args:
    a: 13
    b: 25
  add (call_gTVTWWmtagBEf6nscDkp76m7)
 Call ID: call_gTVTWWmtagBEf6nscDkp76m7
  Args:
    a: 89
    b: 100
================================= Tool Message =================================
Name: add

189
================================== Ai Message ==================================

Here are the results:

- The sum of 13 and 25 is 38.
- The integral of \(\log x\) (assuming natural logarithm, log base \(e\)) is \(x \log x - x + C\), where \(C\) is the constant of integration.
- The sum of 89 and 100 is 189.


In [19]:
!pip install langgraph
!pip install langchain_community
!pip install langchain_openai
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from typing import TypedDict,Sequence,Union

from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("ai_agents_openai")

@tool
def add(a: int, b: int) -> int:
  """Addition tool that adds two numbers together."""
  return a + b

@tool
def multiply(a: int, b: int) -> int:
  """Addition tool that adds two numbers together."""
  return a * b

@tool
def subtract(a: int, b: int) -> int:
  """Addition tool that adds two numbers together."""
  return a - b

tools=[add,multiply,subtract]
model=ChatOpenAI(model="gpt-4o").bind_tools(tools)


class AgentState(TypedDict):
  messages: Annotated[Sequence[BaseMessage], add_messages]

def call_llm(state:AgentState)->AgentState:
  system_prompt = SystemMessage(content= "You are my AI assistant, please answer my query to the best of your ability." )
  response = model.invoke([system_prompt] + state["messages"])
  return {"messages": [response]}

graph = StateGraph(AgentState)
graph.add_node("llm",call_llm)
toolnode=ToolNode(tools)
graph.add_node("tools",toolnode)
graph.set_entry_point("llm")
graph.add_conditional_edges("llm",
                            should_continue,
                           {"continue":"tools","end":END}

)
graph.add_edge("tools", "llm")

app=graph.compile()

inputs={"messages":[("user","capital of netherlands , Add 13 to 25,Multiply 20 with 15 , Subtract 100 from 200 , Integral of logx, 89+100")]}

msg=app.stream(inputs,stream_mode="values")

for s in msg:
  message=s["messages"][-1]
  if isinstance(message,tuple):
    print(message)
  else:
    message.pretty_print()






================================ Human Message =================================

capital of netherlands , Add 13 to 25,Multiply 20 with 15 , Subtract 100 from 200 , Integral of logx, 89+100
================================== Ai Message ==================================
Tool Calls:
  add (call_NGqg8kT6ZJvZ3nYUrwOMQwUH)
 Call ID: call_NGqg8kT6ZJvZ3nYUrwOMQwUH
  Args:
    a: 13
    b: 25
  multiply (call_Cy3iL1CuTe45Qdel0s4NnpZG)
 Call ID: call_Cy3iL1CuTe45Qdel0s4NnpZG
  Args:
    a: 20
    b: 15
  subtract (call_UTWH75F8ADF5zyrSfdUWju1F)
 Call ID: call_UTWH75F8ADF5zyrSfdUWju1F
  Args:
    a: 200
    b: 100
  add (call_XulqGEQ8Bp41ZRAaOzctGfTF)
 Call ID: call_XulqGEQ8Bp41ZRAaOzctGfTF
  Args:
    a: 89
    b: 100
================================= Tool Message =================================
Name: add

189
================================== Ai Message ==================================

- The capital of the Netherlands is Amsterdam.
- 13 added to 25 is 38.
- 20 multiplied by 15 is 300.
